# Practice 12 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: MultiIndex
A DataFrame of quarterly regional sales has been created for you.

1. Set a MultiIndex from `region` and `quarter`, then sort it
2. Retrieve all rows for the `'West'` region
3. Retrieve the single row for `('East', 'Q3')`
4. Find mean revenue per region using NumPy

In [86]:
# Starter data — don't change this
np.random.seed(3)
regions  = ['North', 'East', 'South', 'West']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
sales = pd.DataFrame({
    'region':  regions * 4,
    'quarter': [q for q in quarters for _ in regions],
    'revenue': np.random.randint(50_000, 200_000, size=16),
    'units':   np.random.randint(100, 500, size=16),
})

# Your code here
sales = sales.set_index(['region','quarter'])
sales
sales.sort_index()
ws = sales.loc['West']
sales.loc[('East','Q3')]
for region in sales.index.get_level_values('region').unique():
    print(region, np.mean(sales.loc[region, 'revenue']))

sales.loc['Q3']



North 111294.0
East 117937.5
South 159675.25
West 123036.0


KeyError: 'Q3'

In [ ]:
sales.sort_values(['region','quarter'])
ws = sales[sales['region']=='West']
ss = sales[(sales['region']=='West')&(sales['quarter']=='Q3')]
sales.groupby('region')['revenue'].mean()

### Task 2: Duplicates and Counts
A DataFrame of customer transactions has been created for you.

1. Find how many fully duplicate rows exist
2. Remove duplicates, keeping the first occurrence
3. Find which `customer_id` values appear more than once
4. Show a count of transactions per `category`, sorted highest to lowest

In [67]:
# Starter data — don't change this
orders = pd.DataFrame({
    'order_id':    [1, 2, 3, 2, 4, 5, 5, 6, 7, 3],
    'customer_id': ['C1', 'C2', 'C3', 'C2', 'C1', 'C4', 'C4', 'C5', 'C1', 'C3'],
    'category':    ['Electronics', 'Clothing', 'Food', 'Clothing', 'Electronics',
                    'Food', 'Food', 'Electronics', 'Clothing', 'Food'],
    'amount':      [250, 45, 12, 45, 180, 8, 8, 300, 60, 12],
})

# Your code here
orders.duplicated().sum()
c = orders['customer_id'].value_counts()>=2
c[c].index
orders['category'].value_counts().sort_values(ascending=False)

category
Food           4
Electronics    3
Clothing       3
Name: count, dtype: int64

---
## Level 2 — Transformations

### Task 3: Custom Column Logic
An employee DataFrame has been created for you.

1. Add a `grade` column: `'A'` if `performance_score` ≥ 85, `'B'` if ≥ 70, otherwise `'C'`
2. Add a `bonus` column: 10% of salary for grade A, 5% for B, 0 for C
3. Add a `log_salary` column using NumPy
4. Find the total bonus payout across all employees

In [68]:
# Starter data — don't change this
np.random.seed(7)
employees = pd.DataFrame({
    'name':              ['Alice', 'Bob', 'Carol', 'Dave', 'Eve',
                          'Frank', 'Grace', 'Hank', 'Iris', 'Jake'],
    'department':        np.random.choice(['Eng', 'Sales', 'HR'], size=10),
    'salary':            np.random.randint(40_000, 120_000, size=10),
    'performance_score': np.random.randint(55, 100, size=10),
})

# Your code here

employees['grade'] = pd.cut(employees['performance_score'], 
                            bins=[-np.inf,70,85,np.inf],
                            labels=['C','B','A'],
                            right=False)

bm = {'A': 0.1,
      'B': 0.05,
      'C': 0}
employees['bonus_pct'] = employees['grade'].map(bm).astype(float)
employees['bonus'] = employees['salary'] * employees['bonus_pct']
employees['log_salary'] = np.log(employees['salary'])
employees['bonus'].sum()

np.float64(22243.6)

### Task 4: Reshaping
A wide-format DataFrame of monthly revenue by product has been created for you.

1. Reshape it to long format with columns: `product`, `month`, `revenue`
2. Find the product with the highest total revenue
3. Find the month with the highest total revenue across all products

In [39]:
# Starter data — don't change this
products = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C', 'Widget D'],
    'Jan': [12000,  8500, 15000, 6200],
    'Feb': [13500,  9200, 14000, 7100],
    'Mar': [11000, 10500, 16500, 8300],
    'Apr': [14200,  9800, 13200, 7800],
    'May': [15800, 11200, 17000, 9100],
    'Jun': [13000, 10800, 15500, 8600],
})

# Your code here
m = pd.melt(products, id_vars=['product'],
            value_vars=['Jan','Feb', 'Mar', 'Apr', 'May', 'Jun'],
            var_name="month", value_name='revenue')
m.groupby('product')['revenue'].sum().idxmax()
m.groupby('month')['revenue'].sum().idxmax()
#m


'May'

---
## Level 3 — Aggregation

### Task 5: Rolling Windows
A DataFrame of daily stock prices has been created for you.

1. Compute a 7-day rolling mean and rolling standard deviation of `close_price`
2. Add a `signal` column — `True` when `close_price` is above its 7-day rolling mean
3. Count the number of `True` signals
4. Find the date where the gap between `close_price` and its rolling mean was largest

In [48]:
# Starter data — don't change this
np.random.seed(42)
n = 90
stock = pd.DataFrame({
    'date':        pd.date_range('2024-01-01', periods=n, freq='B'),
    'close_price': np.round(100 + np.cumsum(np.random.randn(n) * 2), 2),
})

# Your code here
stock=stock.set_index('date')
stock['close_price'].rolling(7).mean()
stock['close_price'].rolling(7).std()
stock['signal'] = stock['close_price']>stock['close_price'].rolling(7).mean()
stock['signal'].sum()
gap = stock['close_price'] - stock['close_price'].rolling(7).mean()
gap.idxmax()


Timestamp('2024-04-11 00:00:00')

### Task 6: Multi-level groupby
An e-commerce DataFrame has been created for you.

1. Find total and mean revenue by `region` and `category`
2. For each region, find the category with the highest total revenue
3. Add a `revenue_pct_of_region` column — each row's revenue as a percentage of its region's total
4. Find the 10th and 90th percentile of `revenue` using NumPy

In [69]:
# Starter data — don't change this
np.random.seed(11)
ecommerce = pd.DataFrame({
    'order_id': range(1, 51),
    'region':   np.random.choice(['North', 'South', 'East', 'West'], size=50),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Sports'], size=50),
    'revenue':  np.round(np.random.uniform(10, 500, size=50), 2),
})

# Your code here
ecommerce.groupby(['region','category'])['revenue'].agg(['sum','mean'])
ecommerce.groupby(['region','category'])['revenue'].sum().groupby(level=0).idxmax()
ecommerce['rt'] = ecommerce.groupby('region')['revenue'].transform('sum')
ecommerce['revenue_pct_of_region'] = ecommerce['revenue']/ecommerce['rt']
ecommerce

p10, p90 = np.percentile(ecommerce['revenue'], [10, 90])


---
## Level 4 — Real-world

### Task 7: Full Pipeline
Load the penguins dataset and run a full analysis:

1. Drop rows with any missing values
2. Add a `bill_ratio` column = `bill_length_mm / bill_depth_mm`
3. Find mean and std of all numeric columns grouped by `species`
4. Find the species with the highest mean `bill_ratio`
5. Bin `body_mass_g` into 3 groups labelled `['Light', 'Medium', 'Heavy']`, store as `size_group`
6. Build a pivot table of mean `bill_ratio` for each `species` × `island` combination
7. Find the correlation between `flipper_length_mm` and `body_mass_g` using NumPy

In [76]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
penguins = pd.read_csv(url)

# Your code here
penguins = penguins.dropna()
penguins['bill_ratio'] = penguins['bill_length_mm']/penguins['bill_depth_mm']
numeric_cols = penguins.select_dtypes(include='number').columns
penguins.groupby('species')[numeric_cols].agg(['mean', 'std'])
penguins.groupby('species').agg(['mean', 'std'])


TypeError: agg function failed [how->mean,dtype->object]

In [ ]:


penguins.groupby('species')['bill_ratio'].mean().idxmax()
penguins['size_group'] = pd.cut(penguins['body_mass_g'],3,labels=['Light','Medium','Heavy'])
pt = penguins.pivot_table(index='species',columns='island',values='bill_ratio')
np.corrcoef(penguins['flipper_length_mm'], penguins['body_mass_g'])


island,Biscoe,Dream,Torgersen
species,,,
Adelie,1.0,1.0,1.0
Chinstrap,NaN,1.0,NaN
Gentoo,1.0,NaN,NaN
